In [70]:
import pandas as pd

def load_data(url):
    """Load a CSV dataset from a URL."""
    if not url:
        raise ValueError("The data URL cannot be empty.")

    try:
        return pd.read_csv(url)
    except Exception as exc:
        raise ValueError(
            f"Unable to load data from URL: {url}"
        ) from exc


def load_historical_data(url):
    """Load historical hospital admission data."""
    return load_data(url)


def load_planning_data(url):
    """Load hospital planning data."""
    return load_data(url)


def load_readmission_history(url):
    """Load historical readmission data."""
    return load_data(url)

In [71]:
DATA_URL_HISTO= "https://drive.google.com/uc?export=download&id=1YkRYSkAHeH7nMgO-ums8zbU7paODGs3t"
DATA_URL_PLANNING = "https://drive.google.com/uc?export=download&id=1gUYPdg1Xo7avowdMGEjd0fteviFeOIav"
DATA_URL_READ_HISTO = "https://drive.google.com/uc?export=download&id=1KpsZvV5u6p8zyWUuFUR4Iwh1nFwb6GKu"

In [ ]:
df_historical = load_historical_data(DATA_URL_HISTO)
df_historical.shape

(8000, 17)

In [73]:
df_planning = load_planning_data(DATA_URL_PLANNING)
df_planning.shape

(8000, 15)

In [74]:
df_read_histo = load_readmission_history(DATA_URL_READ_HISTO)
df_read_histo.shape

(6183, 18)

In [ ]:
from datetime import timedelta

import pandas as pd

DEFAULT_DATE = "2023-07-01" #Date par defaut pour démarrer le streamlit premier jour hors donnée d'entrainement
DEFAULT_HORIZON = 7 #nombre de jour par défaut pour la vision future
CAPACITY = 80 # Valeur par défaut ajustable ajustable par les utilisateurs
HISTORICAL_WINDOW_DAYS = 60 #fenêtre des données historiques par défaut
REDUCED_VIEW_DAYS = 7
FULL_VIEW_DAYS = 30
DT_READ = 30     


def prepare_forecast_period(
    date_ref,
    forecast_horizon_days,
    reduced_view,
    date_min,
):
    """Build the historical and forecast date ranges."""

    date_ref = pd.Timestamp(date_ref)

    date_end = date_ref + timedelta(
        days=forecast_horizon_days - 1
    )

    view_days = (
        REDUCED_VIEW_DAYS
        if reduced_view
        else FULL_VIEW_DAYS
    )

    date_start = max(
        date_ref - timedelta(days=view_days),
        pd.Timestamp(date_min),
    )

    return date_ref, date_start, date_end


def prepare_historical_data(df_historique, date_ref):
    """Prepare historical admissions for forecasting."""

    start_date = date_ref - timedelta(
        days=HISTORICAL_WINDOW_DAYS
    )

    mask = (
        (pd.to_datetime(df_historique["admission_date"]) >= start_date)
        & (pd.to_datetime(df_historique["admission_date"]) < date_ref)
    )

    df_histo = df_historique.loc[mask].copy()
    df_histo["label"] = 0

    return df_histo


def prepare_readmission_history(
    df_read_histo,
    date_ref,
):
    """Prepare historical readmission data."""

    start_date = date_ref - timedelta(
        days=HISTORICAL_WINDOW_DAYS
    )

    mask = (
        (pd.to_datetime(df_read_histo["admission_date"]) >= start_date)
        & (pd.to_datetime(df_read_histo["admission_date"]) < date_ref)
    )

    return df_read_histo.loc[mask].copy()


def prepare_historical_population(
    df_histo,
    df_read_histo,
):
    """Combine historical admissions and readmissions."""

    return pd.concat(
        [df_histo, df_read_histo],
        ignore_index=True,
    )

In [76]:
date_ref, date_start, date_end = prepare_forecast_period(
    DEFAULT_DATE,
    DEFAULT_HORIZON,
    False,
    "2022-07-01",
)

print(date_ref)
print(date_start)
print(date_end)

2023-07-01 00:00:00
2023-06-01 00:00:00
2023-07-07 00:00:00


In [77]:
df_histo = prepare_historical_data(
    df_historical,
    date_ref
)

In [78]:
df_histo.describe()

,age,comorbidities_count,length_of_stay,medications_count,followup_visits_last_year,prev_readmissions,readmission_risk_score,label
count,458.000000,458.000000,458.000000,458.000000,458.000000,458.000000,458.000000,458.0
mean,56.668122,4.248908,7.290393,7.338428,3.506550,1.530568,0.755284,0.0
std,17.278359,1.403048,1.867831,2.339156,1.599015,0.870212,0.235332,0.0
min,18.000000,1.000000,3.000000,2.000000,0.000000,0.000000,0.070000,0.0
25%,45.000000,3.000000,6.000000,6.000000,2.000000,1.000000,0.590000,0.0
50%,57.000000,4.000000,7.000000,8.000000,3.000000,1.000000,0.815000,0.0
75%,69.000000,5.000000,9.000000,8.000000,4.000000,2.000000,0.970000,0.0
max,95.000000,9.000000,14.000000,16.000000,9.000000,5.000000,0.970000,0.0
